In [1]:
import numpy              as np
import matplotlib.pyplot  as plt
import seaborn            as sns
import libraries.plotting as slp
import os

from pymatgen.io.vasp.outputs import Vasprun

sns.set_theme()

First we compute linear macroscopic average potential for bulk and each slab. Then we compute the band alignment for each slab.

The linear macroscopic average potential is computed as:

$echo -e "426" | ~/vaspkit.1.3.5/bin/vaspkit

In [ ]:
# Define name to reference folder which stores the energies of each slab
slab_folder = 'input/slabs/BiSeI-slab'

# Ratio do discard from the edges for averaging
discard_ratio = 0.25

# Bulk

In [ ]:
bulk_folder = f'{slab_folder}/bulk'

linear_mac_pot = np.loadtxt(f'{bulk_folder}/PLANAR_AVERAGE.dat')

In [ ]:
# Load vasprun
vasprun = Vasprun(f'{bulk_folder}/vasprun.xml')

# Get band structure
band_structure = vasprun.get_band_structure()

# Get valence band maximum and band gap
valence_band_DFT = band_structure.get_vbm()["energy"]
band_gap         = band_structure.get_band_gap()["energy"]

In [ ]:
energy_section = linear_mac_pot[:, 1]

mac_avg_pot_bulk = np.mean(energy_section)

np.savetxt(f'{bulk_folder}/macroscopic_average_potential', [mac_avg_pot_bulk])

print(f'Mean value: {mac_avg_pot_bulk}')

In [ ]:
plt.plot(linear_mac_pot[:, 0], linear_mac_pot[:, 1], label='Linear average potential')

plt.savefig(f'{bulk_folder}/macroscopic_average_potential.eps', bbox_inches='tight')
plt.show()

# Slab

In [ ]:
# Iterate over slabs
for miller_index_str in os.listdir(slab_folder):
    # Define current folder
    miller_folder = f'{slab_folder}/{miller_index_str}'

    # Skip in case it is not a folder or it is the bulk folder
    if (not os.path.isdir(miller_folder)) or (miller_index_str == 'bulk'):
        continue

    print(miller_folder)

    # Load planar average potential
    linear_mac_pot = np.loadtxt(f'{miller_folder}/PLANAR_AVERAGE.dat')

    # Divide into bulk and vacuum
    mid_idx = int(0.5*len(linear_mac_pot))

    bulk_mac_pot   = linear_mac_pot[:mid_idx]
    vacuum_mac_pot = linear_mac_pot[mid_idx:]

    # Bulk
    idx_ref = int(discard_ratio*len(bulk_mac_pot))
    bulk_valid_mac_pot = bulk_mac_pot[idx_ref:-idx_ref]
    mean_avg_pot_bulk = np.mean(bulk_valid_mac_pot[: ,1])

    print(f'\tPotential in bulk: {mean_avg_pot_bulk}')
    plt.plot(bulk_valid_mac_pot[:, 0], bulk_valid_mac_pot[:, 1], label='Bulk')
    plt.plot([bulk_valid_mac_pot[0, 0], bulk_valid_mac_pot[-1, 0]], [mean_avg_pot_bulk, mean_avg_pot_bulk])

    # Vacuum
    idx_ref = int(discard_ratio*len(vacuum_mac_pot))
    vacuum_valid_mac_pot = vacuum_mac_pot[idx_ref:-idx_ref]
    mean_avg_pot_vacuum = np.mean(vacuum_valid_mac_pot[: ,1])

    print(f'\tPotential in vacuum: {mean_avg_pot_vacuum}')
    plt.plot(vacuum_valid_mac_pot[:, 0], vacuum_valid_mac_pot[:, 1], label='Vacuum')
    plt.plot([vacuum_valid_mac_pot[0, 0], vacuum_valid_mac_pot[-1, 0]], [mean_avg_pot_vacuum, mean_avg_pot_vacuum])

    # Save the data
    np.savetxt(f'{miller_folder}/macroscopic_average_potential', [mean_avg_pot_bulk, mean_avg_pot_vacuum])

    plt.savefig(f'{miller_folder}/macroscopic_average_potential.eps', bbox_inches='tight')
    plt.show()
    
    # Compute band alignments referred to vacuum
    valence_band    = valence_band_DFT + mean_avg_pot_bulk - mac_avg_pot_bulk + 0 - mean_avg_pot_vacuum
    conduction_band = valence_band + band_gap

    # Save the data
    np.savetxt(f'{miller_folder}/valence_band',    [valence_band])
    np.savetxt(f'{miller_folder}/conduction_band', [conduction_band])

The position of the valence band with respect to vacuum level is computed as:

\begin{equation}
    VBM = VMB_{DFT} - V_{bulk} + (V_{slab} - V_{vacuum})
\end{equation}